# BA830 – Scarcity Messaging Experiment
## Data Cleaning & Preparation

This notebook loads the raw Qualtrics export, cleans it, and produces an
analysis-ready dataset at both the **respondent level** and the
**respondent × product (long) level**.

In [18]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
import statsmodels.formula.api as smf
import pyfixest as pf

## 1. Load export, Clean, & Preprocess

In [2]:
raw = pd.read_csv("BA830_Survey.csv")

# Row 0 is question text – save it then drop
question_text = raw.iloc[0].to_dict()
df = raw.iloc[1:].reset_index(drop=True)

print(f"Raw responses: {len(df)}")
df.head()

Raw responses: 99


,ResponseId,Finished,Q1,Q2,Q3,Q4,Q5,Q6,Q7,Q8,...,QID73,QID76,QID77,QID78,QID79,QID82,QID83,QID84,QID85,Unnamed: 34
0,R_3lsKvK0Jg664xnd,TRUE,26+,Female,Graduate,Moderately price-sensitive,Often ( 1-2 times a week),Very impulsive,Strongly agree,Neither agree nor disagree,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,R_3g2e2mgxFgQdnYz,TRUE,26+,Non-binary / third gender,Undergraduate,Moderately price-sensitive,Sometimes (1-2 times a month),Moderately impulsive,Somewhat Disagree,Neither agree nor disagree,...,Only 3 left in stock,Extremely unlikely,Not urgent at all,None at all,Only 3 left in stock,Extremely unlikely,Not urgent at all,None at all,Only 3 left in stock,NaN
2,R_5QYe0WkAQuDWk5E,TRUE,22-25,Male,Graduate,Moderately price-sensitive,Sometimes (1-2 times a month),Not at all impulsive,Somewhat Disagree,Somewhat disagree,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,R_6cvLdx8OHhwIEAU,TRUE,22-25,Female,Graduate,Moderately price-sensitive,Often ( 1-2 times a week),Moderately impulsive,Somewhat Agree,Somewhat agree,...,Only 3 left in stock,Extremely unlikely,Not urgent at all,None at all,Only 3 left in stock,Extremely unlikely,Not urgent at all,None at all,Only 3 left in stock,NaN
4,R_3EnYo2qByLAB1JO,TRUE,22-25,Male,Graduate,Very price-sensitive,Rarely (a few times a year),Not at all impulsive,Strongly disagree,Somewhat agree,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Filter to usable responses

In [3]:
# Drop trailing empty column
df = df.loc[:, ~df.columns.str.startswith("Unnamed")]

# Keep only completed surveys in the target age range
df = df[df["Finished"] == "TRUE"].copy()
df = df[df["Q1"].isin(["18-21", "22-25"])].copy()

print(f"Usable responses: {len(df)}")
print(f"  18-21: {(df['Q1']=='18-21').sum()}")
print(f"  22-25: {(df['Q1']=='22-25').sum()}")

Usable responses: 60
  18-21: 18
  22-25: 42


### Rename demographics & encode covariates

In [4]:
# Readable column names for demographics
df = df.rename(columns={
    "ResponseId": "response_id",
    "Q1": "age_range",
    "Q2": "gender",
    "Q3": "student_status",
    "Q4": "price_sensitivity",
    "Q5": "online_purchase_freq",
    "Q6": "impulsive_rating",
    "Q7": "impulse_agreement",
    "Q8": "scarcity_agreement",
})

# --- Numeric encodings for covariates ---
price_sens_map = {
    "Slightly price-sensitive": 1,
    "Moderately price-sensitive": 2,
    "Very price-sensitive": 3,
    "Extremely price-sensitive": 4,
}

purchase_freq_map = {
    "Rarely (a few times a year)": 1,
    "Sometimes (1-2 times a month)": 2,
    "Often ( 1-2 times a week)": 3,
    "Always (3+ times a week)": 4,
}

impulsive_map = {
    "Not at all impulsive": 1,
    "Slightly impulsive": 2,
    "Moderately impulsive": 3,
    "Very impulsive": 4,
    "Extremely impulsive": 5,
}

agree5_map = {
    "Strongly disagree": 1, "Strongly Disagree": 1,
    "Somewhat disagree": 2, "Somewhat Disagree": 2,
    "Neither agree nor disagree": 3, "Neither Agree nor Disagree": 3,
    "Somewhat agree": 4, "Somewhat Agree": 4,
    "Strongly agree": 5, "Strongly Agree": 5,
}

df["price_sensitivity_num"] = df["price_sensitivity"].map(price_sens_map)
df["purchase_freq_num"] = df["online_purchase_freq"].map(purchase_freq_map)
df["impulsive_num"] = df["impulsive_rating"].map(impulsive_map)
df["impulse_agree_num"] = df["impulse_agreement"].map(agree5_map)
df["scarcity_agree_num"] = df["scarcity_agreement"].map(agree5_map)

# Binary dummies useful for regressions
df["is_female"] = (df["gender"] == "Female").astype(int)
df["is_grad"] = (df["student_status"] == "Graduate").astype(int)
df["age_22_25"] = (df["age_range"] == "22-25").astype(int)

print("Covariate summary:")
df[["price_sensitivity_num", "purchase_freq_num", "impulsive_num",
    "impulse_agree_num", "scarcity_agree_num"]].describe().round(2)

Covariate summary:


,price_sensitivity_num,purchase_freq_num,impulsive_num,impulse_agree_num,scarcity_agree_num
count,60.00,60.00,60.00,60.00,60.00
mean,2.28,2.25,2.33,2.92,3.28
std,0.76,0.70,0.93,1.14,1.12
min,1.00,1.00,1.00,1.00,1.00
25%,2.00,2.00,2.00,2.00,2.00
50%,2.00,2.00,2.00,3.00,4.00
75%,3.00,3.00,3.00,4.00,4.00
max,4.00,4.00,5.00,5.00,5.00


### Assign treatment condition & merge product branches

Qualtrics randomized each respondent into one of two branches:
- **Branch A (control):** QID45/50/55 → products shown *without* scarcity messaging
- **Branch B (treatment):** QID70/76/82 → products shown *with* scarcity messaging

Only one branch is populated per respondent, so we coalesce them into unified `p1_*`, `p2_*`, `p3_*` columns.

In [5]:
# Condition: control if Branch A (QID45) is filled, else treatment
df["condition"] = np.where(df["QID45"].notna(), "control", "treatment")
df["treat"] = (df["condition"] == "treatment").astype(int)

# Coalesce Branch A and Branch B into unified product columns
branch_map = {
    # (branch_a, branch_b) -> unified name
    # Product 1
    ("QID45", "QID70"): "p1_likelihood",
    ("QID46", "QID71"): "p1_urgency",
    ("QID20", "QID72"): "p1_regret",
    ("QID21", "QID73"): "p1_avail_msg",
    # Product 2
    ("QID50", "QID76"): "p2_likelihood",
    ("QID51", "QID77"): "p2_urgency",
    ("QID52", "QID78"): "p2_regret",
    ("QID53", "QID79"): "p2_avail_msg",
    # Product 3
    ("QID55", "QID82"): "p3_likelihood",
    ("QID56", "QID83"): "p3_urgency",
    ("QID57", "QID84"): "p3_regret",
    ("QID58", "QID85"): "p3_avail_msg",
}

for (col_a, col_b), new_name in branch_map.items():
    df[new_name] = df[col_a].combine_first(df[col_b])

# Drop the raw branch-specific QID columns
raw_qids = [c for pair in branch_map.keys() for c in pair]
df = df.drop(columns=[c for c in raw_qids if c in df.columns])

print(f"Condition counts:")
print(df["condition"].value_counts())

Condition counts:
condition
treatment    33
control      27
Name: count, dtype: int64


### Encode outcome variables (Likert → numeric)

In [12]:
likelihood_map = {
    "Extremely unlikely": 1,
    "Somewhat unlikely": 2,
    "Neither likely nor unlikely": 3,
    "Somewhat likely": 4,
    "Extremely likely": 5,
}

urgency_map = {
    "Not urgent at all": 1,
    "Slightly urgent": 2,
    "Moderately urgent": 3,
    "Very urgent": 4,
    "Extremely urgent": 5,
    "Extremely Urgent": 5,  # handle capitalization variant
}

regret_map = {
    "None at all": 1,
    "A little": 2,
    "A moderate amount": 3,
    "A lot": 4,
    "A great deal": 5,
}

for p in ["p1", "p2", "p3"]:
    df[f"{p}_likelihood_num"] = df[f"{p}_likelihood"].map(likelihood_map)
    df[f"{p}_urgency_num"] = df[f"{p}_urgency"].map(urgency_map)
    df[f"{p}_regret_num"] = df[f"{p}_regret"].map(regret_map)
    df[f"{p}_scarcity_msg"] = (
        df[f"{p}_avail_msg"].str.contains("only|left", case=False, na=False).astype(int)
    )

# Respondent-level averages across the 3 products
df["avg_likelihood"] = df[["p1_likelihood_num", "p2_likelihood_num", "p3_likelihood_num"]].mean(axis=1)
df["avg_urgency"] = df[["p1_urgency_num", "p2_urgency_num", "p3_urgency_num"]].mean(axis=1)
df["avg_regret"] = df[["p1_regret_num", "p2_regret_num", "p3_regret_num"]].mean(axis=1)

print("Outcome means by condition:")
print(df.groupby("condition")[["avg_likelihood", "avg_urgency", "avg_regret"]].mean().round(3))

Outcome means by condition:
           avg_likelihood  avg_urgency  avg_regret
condition                                         
control             2.531        1.889       1.642
treatment           2.606        2.141       1.859


### Build (respondent × product) panel

Stacking 3 rows per respondent lets us run product fixed-effects models
and increases our effective sample from 60 to 180.

In [7]:
# Columns that are the same across products (respondent-level)
id_cols = [
    "response_id", "condition", "treat",
    "age_range", "gender", "student_status",
    "price_sensitivity_num", "purchase_freq_num", "impulsive_num",
    "impulse_agree_num", "scarcity_agree_num",
    "is_female", "is_grad", "age_22_25",
]

long_frames = []
for i, p in enumerate(["p1", "p2", "p3"], start=1):
    chunk = df[id_cols + [
        f"{p}_likelihood_num", f"{p}_urgency_num",
        f"{p}_regret_num", f"{p}_scarcity_msg",
    ]].copy()
    chunk = chunk.rename(columns={
        f"{p}_likelihood_num": "likelihood",
        f"{p}_urgency_num": "urgency",
        f"{p}_regret_num": "regret",
        f"{p}_scarcity_msg": "scarcity_msg",
    })
    chunk["product"] = i
    long_frames.append(chunk)

df_long = pd.concat(long_frames, ignore_index=True)

print(f"Long dataset: {len(df_long)} rows  ({len(df)} respondents × 3 products)")
print()
print("Means by condition (long):")
print(df_long.groupby("condition")[["likelihood", "urgency", "regret"]].mean().round(3))

Long dataset: 180 rows  (60 respondents × 3 products)

Means by condition (long):
           likelihood  urgency  regret
condition                             
control         2.531    1.889   1.642
treatment       2.606    2.141   1.859


### Covariate balance check

In [ ]:
balance_vars = [
    "age_22_25", "is_female", "is_grad",
    "price_sensitivity_num", "purchase_freq_num",
    "impulsive_num", "impulse_agree_num", "scarcity_agree_num",
]

balance = df.groupby("condition")[balance_vars].mean().T
balance["diff"] = balance["treatment"] - balance["control"]
print(balance.round(3))

condition              control  treatment   diff
age_22_25                0.630      0.758  0.128
is_female                0.407      0.364 -0.044
is_grad                  0.593      0.636  0.044
price_sensitivity_num    2.074      2.455  0.380
purchase_freq_num        2.259      2.242 -0.017
impulsive_num            2.037      2.576  0.539
impulse_agree_num        2.519      3.242  0.724
scarcity_agree_num       3.444      3.152 -0.293


### Save cleaned datasets

In [9]:
out = Path(".")

df.to_csv(out / "BA830_Survey_clean_wide.csv", index=False)
df_long.to_csv(out / "BA830_Survey_clean_long.csv", index=False)

# Save codebook
codebook = pd.DataFrame({
    "original_column": raw.columns,
    "question_text": [question_text.get(c, "") for c in raw.columns],
})
codebook.to_csv(out / "BA830_Survey_codebook.csv", index=False)

print("Saved:")
print(f"  Wide:     {out / 'BA830_Survey_clean_wide.csv'}  ({df.shape})")
print(f"  Long:     {out / 'BA830_Survey_clean_long.csv'}  ({df_long.shape})")
print(f"  Codebook: {out / 'BA830_Survey_codebook.csv'}")

Saved:
  Wide:     BA830_Survey_clean_wide.csv  ((60, 47))
  Long:     BA830_Survey_clean_long.csv  ((180, 19))
  Codebook: BA830_Survey_codebook.csv


## 2. Analysis

### 2a. T-Test: Difference In Means

In [ ]:
results = []
for outcome in ["avg_likelihood", "avg_urgency", "avg_regret"]:
    ctrl = df.loc[df["treat"] == 0, outcome].dropna()
    trt  = df.loc[df["treat"] == 1, outcome].dropna()
    diff = trt.mean() - ctrl.mean()
    t_stat, p_val = stats.ttest_ind(trt, ctrl, equal_var=False)
    results.append({
        "Outcome": outcome.replace("avg_", "").title(),
        "Control Mean": round(ctrl.mean(), 3),
        "Treatment Mean": round(trt.mean(), 3),
        "Difference": round(diff, 3),x
        "t-stat": round(t_stat, 3),
        "p-value": round(p_val, 3),})

pd.DataFrame(results).set_index("Outcome")

,Control Mean,Treatment Mean,Difference,t-stat,p-value
Outcome,,,,,
Likelihood,2.531,2.606,0.075,0.367,0.715
Urgency,1.889,2.141,0.253,1.291,0.202
Regret,1.642,1.859,0.217,0.958,0.342


## 2b. OLS - Average Treatment Effect 

Formula used:
$$outcome_i = \beta_0 + \beta_1 \cdot treat_i + \varepsilon_i$$

In [19]:
reg_likelihood = pf.feols('avg_likelihood ~ treat', data=df, vcov='hetero')
reg_urgency = pf.feols('avg_urgency ~ treat', data=df, vcov='hetero')
reg_regret = pf.feols('avg_regret ~ treat', data=df, vcov='hetero')

pf.etable([reg_likelihood, reg_urgency, reg_regret])

GT(_tbl_data=  level_0             level_1                      0                      1  \
0    coef               treat     0.075 <br> (0.205)     0.253 <br> (0.196)   
1    coef           Intercept  2.531*** <br> (0.159)  1.889*** <br> (0.139)   
2   stats        Observations                     60                     60   
3   stats           S.E. type                 hetero                 hetero   
4   stats       R<sup>2</sup>                  0.002                  0.027   
5   stats  Adj. R<sup>2</sup>                 -0.015                  0.011   

                       2  
0     0.217 <br> (0.226)  
1  1.642*** <br> (0.166)  
2                     60  
3                 hetero  
4                  0.016  
5                 -0.001  , _body=<great_tables._gt_data.Body object at 0x1198c6660>, _boxhead=Boxhead([ColInfo(var='level_0', type=<ColInfoTypeEnum.row_group: 3>, column_label='level_0', column_align='center', column_width=None), ColInfo(var='level_1', type=<ColInfoTypeEnum.stub: 2>, column_label='level_1', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label='(1)', column_align='center', column_width=None), ColInfo(var='1', type=<ColInfoTypeEnum.default: 1>, column_label='(2)', column_align='center', column_width=None), ColInfo(var='2', type=<ColInfoTypeEnum.default: 1>, column_label='(3)', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x11970a990>, _spanners=Spanners([SpannerInfo(spanner_id='avg_likelihood', spanner_level=1, spanner_label='avg_likelihood', spanner_units=None, spanner_pattern=None, vars=['0'], built=None), SpannerInfo(spanner_id='avg_urgency', spanner_level=1, spanner_label='avg_urgency', spanner_units=None, spanner_pattern=None, vars=['1'], built=None), SpannerInfo(spanner_id='avg_regret', spanner_level=1, spanner_label='avg_regret', spanner_units=None, spanner_pattern=None, vars=['2'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x119918980>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x11970b4d0>, _source_notes=['Significance levels: * p < 0.05, ** p < 0.01, *** p < 0.001. Format of coefficient cell:\nCoefficient \n (Std. Error)'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x119919010>, _formats=[], _substitutions=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table

## 2c. OLS with Covariates (Precision Boost)

$$outcome_i = \beta_0 + \beta_1 \cdot treat_i + \gamma' X_i + \varepsilon_i$$

Adding covariates absorbs residual variance meaning tighter standard errors on β₁. Randomization means β₁ stays causal regardless.

In [20]:
covs = 'impulsive_num + price_sensitivity_num + purchase_freq_num + is_female + age_22_25 + is_grad'

reg_lik_cov = pf.feols(f'avg_likelihood ~ treat + {covs}', data=df, vcov='hetero')
reg_urg_cov = pf.feols(f'avg_urgency ~ treat + {covs}', data=df, vcov='hetero')
reg_reg_cov = pf.feols(f'avg_regret ~ treat + {covs}', data=df, vcov='hetero')

pf.etable([reg_lik_cov, reg_urg_cov, reg_reg_cov])

GT(_tbl_data=   level_0                level_1                      0  \
0     coef                  treat     0.207 <br> (0.219)   
1     coef          impulsive_num    -0.035 <br> (0.144)   
2     coef  price_sensitivity_num   -0.281* <br> (0.137)   
3     coef      purchase_freq_num     0.067 <br> (0.180)   
4     coef              is_female    -0.118 <br> (0.208)   
5     coef              age_22_25    -0.172 <br> (0.205)   
6     coef                is_grad     0.275 <br> (0.201)   
7     coef              Intercept  3.027*** <br> (0.591)   
8    stats           Observations                     60   
9    stats              S.E. type                 hetero   
10   stats          R<sup>2</sup>                  0.113   
11   stats     Adj. R<sup>2</sup>                 -0.007   

                        1                    2  
0      0.275 <br> (0.204)   0.101 <br> (0.256)  
1      0.102 <br> (0.184)   0.229 <br> (0.181)  
2     -0.132 <br> (0.117)  -0.052 <br> (0.152)  
3      0.120 <br> (0.215)   0.020 <br> (0.223)  
4     -0.120 <br> (0.183)  -0.140 <br> (0.211)  
5     -0.108 <br> (0.189)   0.222 <br> (0.309)  
6    -0.373* <br> (0.177)  -0.504 <br> (0.295)  
7   2.023*** <br> (0.537)  1.455* <br> (0.684)  
8                      60                   60  
9                  hetero               hetero  
10                  0.154                0.141  
11                  0.040                0.026  , _body=<great_tables._gt_data.Body object at 0x119904c30>, _boxhead=Boxhead([ColInfo(var='level_0', type=<ColInfoTypeEnum.row_group: 3>, column_label='level_0', column_align='center', column_width=None), ColInfo(var='level_1', type=<ColInfoTypeEnum.stub: 2>, column_label='level_1', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label='(1)', column_align='center', column_width=None), ColInfo(var='1', type=<ColInfoTypeEnum.default: 1>, column_label='(2)', column_align='center', column_width=None), ColInfo(var='2', type=<ColInfoTypeEnum.default: 1>, column_label='(3)', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x119907950>, _spanners=Spanners([SpannerInfo(spanner_id='avg_likelihood', spanner_level=1, spanner_label='avg_likelihood', spanner_units=None, spanner_pattern=None, vars=['0'], built=None), SpannerInfo(spanner_id='avg_urgency', spanner_level=1, spanner_label='avg_urgency', spanner_units=None, spanner_pattern=None, vars=['1'], built=None), SpannerInfo(spanner_id='avg_regret', spanner_level=1, spanner_label='avg_regret', spanner_units=None, spanner_pattern=None, vars=['2'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x119966490>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x119905cd0>, _source_notes=['Significance levels: * p < 0.05, ** p < 0.01, *** p < 0.001. Format of coefficient cell:\nCoefficient \n (Std. Error)'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x119965a90>, _formats=[], _substitutions=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 

### 2d. Heterogeneous Treatment Effects

Does scarcity messaging work more on impulsive shoppers or price-sensitive shoppers? The interaction term (treat × moderator) tells us.

In [21]:
reg_hte_imp = pf.feols('avg_likelihood ~ treat * impulsive_num', data=df, vcov='hetero')
reg_hte_price = pf.feols('avg_likelihood ~ treat * price_sensitivity_num', data=df, vcov='hetero')
reg_hte_imp_urg = pf.feols('avg_urgency ~ treat * impulsive_num', data=df, vcov='hetero')
reg_hte_price_urg = pf.feols('avg_urgency ~ treat * price_sensitivity_num', data=df, vcov='hetero')

pf.etable([reg_hte_imp, reg_hte_price, reg_hte_imp_urg, reg_hte_price_urg])

GT(_tbl_data=  level_0                      level_1                      0  \
0    coef                        treat    -0.259 <br> (0.578)   
1    coef                impulsive_num    -0.031 <br> (0.210)   
2    coef          treat:impulsive_num     0.136 <br> (0.249)   
3    coef        price_sensitivity_num                          
4    coef  treat:price_sensitivity_num                          
5    coef                    Intercept  2.595*** <br> (0.471)   
6   stats                 Observations                     60   
7   stats                    S.E. type                 hetero   
8   stats                R<sup>2</sup>                  0.012   
9   stats           Adj. R<sup>2</sup>                 -0.041   

                       1                      2                      3  
0    -0.273 <br> (0.541)     0.051 <br> (0.528)    -0.353 <br> (0.535)  
1                            0.124 <br> (0.152)                         
2                            0.052 <br> (0.225)                         
3   -0.427* <br> (0.191)                          -0.319* <br> (0.122)  
4     0.208 <br> (0.234)                            0.296 <br> (0.209)  
5  3.417*** <br> (0.381)  1.635*** <br> (0.337)  2.550*** <br> (0.275)  
6                     60                     60                     60  
7                 hetero                 hetero                 hetero  
8                  0.091                  0.062                  0.063  
9                  0.042                  0.012                  0.012  , _body=<great_tables._gt_data.Body object at 0x119a33680>, _boxhead=Boxhead([ColInfo(var='level_0', type=<ColInfoTypeEnum.row_group: 3>, column_label='level_0', column_align='center', column_width=None), ColInfo(var='level_1', type=<ColInfoTypeEnum.stub: 2>, column_label='level_1', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label='(1)', column_align='center', column_width=None), ColInfo(var='1', type=<ColInfoTypeEnum.default: 1>, column_label='(2)', column_align='center', column_width=None), ColInfo(var='2', type=<ColInfoTypeEnum.default: 1>, column_label='(3)', column_align='center', column_width=None), ColInfo(var='3', type=<ColInfoTypeEnum.default: 1>, column_label='(4)', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x119b71130>, _spanners=Spanners([SpannerInfo(spanner_id='avg_likelihood', spanner_level=1, spanner_label='avg_likelihood', spanner_units=None, spanner_pattern=None, vars=['0', '1'], built=None), SpannerInfo(spanner_id='avg_urgency', spanner_level=1, spanner_label='avg_urgency', spanner_units=None, spanner_pattern=None, vars=['2', '3'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x119904e90>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x119b71370>, _source_notes=['Significance levels: * p < 0.05, ** p < 0.01, *** p < 0.001. Format of coefficient cell:\nCoefficient \n (Std. Error)'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x119964050>, _formats=[], _substitutions=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 

### 2e. Product Fixed Effects + Clustered SEs (Long Panel)

$$outcome_{ip} = \beta_0 + \beta_1 \cdot treat_i + \delta_p + \varepsilon_{ip}$$

Product FE controls for some products being inherently more desirable. SEs clustered at respondent level since each person gives 3 correlated responses.

In [24]:
covs = 'impulsive_num + price_sensitivity_num + purchase_freq_num + is_female + age_22_25 + is_grad'

reg_long_lik = pf.feols('likelihood ~ treat | product', data=df_long, vcov={'CRV1': 'response_id'})
reg_long_lik_cov = pf.feols(f'likelihood ~ treat + {covs} | product', data=df_long, vcov={'CRV1': 'response_id'})
reg_long_urg = pf.feols('urgency ~ treat | product', data=df_long, vcov={'CRV1': 'response_id'})
reg_long_urg_cov = pf.feols(f'urgency ~ treat + {covs} | product', data=df_long, vcov={'CRV1': 'response_id'})
reg_long_reg = pf.feols('regret ~ treat | product', data=df_long, vcov={'CRV1': 'response_id'})
reg_long_reg_cov = pf.feols(f'regret ~ treat + {covs} | product', data=df_long, vcov={'CRV1': 'response_id'})

pf.etable([reg_long_lik, reg_long_lik_cov, reg_long_urg, reg_long_urg_cov, reg_long_reg, reg_long_reg_cov])

GT(_tbl_data=   level_0                level_1                   0                     1  \
0     coef                  treat  0.075 <br> (0.205)    0.207 <br> (0.211)   
1     coef          impulsive_num                       -0.035 <br> (0.139)   
2     coef  price_sensitivity_num                      -0.281* <br> (0.132)   
3     coef      purchase_freq_num                        0.067 <br> (0.173)   
4     coef              is_female                       -0.118 <br> (0.200)   
5     coef              age_22_25                       -0.172 <br> (0.197)   
6     coef                is_grad                        0.275 <br> (0.194)   
7       fe                product                   x                     x   
8    stats           Observations                 180                   180   
9    stats              S.E. type     by: response_id       by: response_id   
10   stats          R<sup>2</sup>               0.127                 0.173   
11   stats   R<sup>2</sup> Within               0.001                 0.053   

                     2                     3                   4  \
0   0.253 <br> (0.196)    0.275 <br> (0.196)  0.217 <br> (0.226)   
1                         0.102 <br> (0.177)                       
2                        -0.132 <br> (0.113)                       
3                         0.120 <br> (0.207)                       
4                        -0.120 <br> (0.177)                       
5                        -0.108 <br> (0.182)                       
6                       -0.373* <br> (0.171)                       
7                    x                     x                   x   
8                  180                   180                 180   
9      by: response_id       by: response_id     by: response_id   
10               0.131                 0.192               0.016   
11               0.015                 0.084               0.010   

                      5  
0    0.101 <br> (0.246)  
1    0.229 <br> (0.174)  
2   -0.052 <br> (0.147)  
3    0.020 <br> (0.215)  
4   -0.140 <br> (0.203)  
5    0.222 <br> (0.298)  
6   -0.504 <br> (0.284)  
7                     x  
8                   180  
9       by: response_id  
10                0.100  
11                0.095  , _body=<great_tables._gt_data.Body object at 0x11564f950>, _boxhead=Boxhead([ColInfo(var='level_0', type=<ColInfoTypeEnum.row_group: 3>, column_label='level_0', column_align='center', column_width=None), ColInfo(var='level_1', type=<ColInfoTypeEnum.stub: 2>, column_label='level_1', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label='(1)', column_align='center', column_width=None), ColInfo(var='1', type=<ColInfoTypeEnum.default: 1>, column_label='(2)', column_align='center', column_width=None), ColInfo(var='2', type=<ColInfoTypeEnum.default: 1>, column_label='(3)', column_align='center', column_width=None), ColInfo(var='3', type=<ColInfoTypeEnum.default: 1>, column_label='(4)', column_align='center', column_width=None), ColInfo(var='4', type=<ColInfoTypeEnum.default: 1>, column_label='(5)', column_align='center', column_width=None), ColInfo(var='5', type=<ColInfoTypeEnum.default: 1>, column_label='(6)', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x11a5da8a0>, _spanners=Spanners([SpannerInfo(spanner_id='likelihood', spanner_level=1, spanner_label='likelihood', spanner_units=None, spanner_pattern=None, vars=['0', '1'], built=None), SpannerInfo(spanner_id='urgency', spanner_level=1, spanner_label='urgency', spanner_units=None, spanner_pattern=None, vars=['2', '3'], built=None), SpannerInfo(spanner_id='regret', spanner_level=1, spanner_label='regret', spanner_units=None, spanner_pattern=None, vars=['4', '5'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x11a5d8230>, _summary_rows_grand=<great_tables._gt_da

### 2f. Heterogeneous Treatment Effects (Long Panel)

In [25]:
reg_long_hte1 = pf.feols('likelihood ~ treat * impulsive_num | product', data=df_long, vcov={'CRV1': 'response_id'})
reg_long_hte2 = pf.feols('urgency ~ treat * impulsive_num | product', data=df_long, vcov={'CRV1': 'response_id'})
reg_long_hte3 = pf.feols('regret ~ treat * impulsive_num | product', data=df_long, vcov={'CRV1': 'response_id'})
reg_long_hte4 = pf.feols('likelihood ~ treat * price_sensitivity_num | product', data=df_long, vcov={'CRV1': 'response_id'})
reg_long_hte5 = pf.feols('urgency ~ treat * price_sensitivity_num | product', data=df_long, vcov={'CRV1': 'response_id'})
reg_long_hte6 = pf.feols('regret ~ treat * price_sensitivity_num | product', data=df_long, vcov={'CRV1': 'response_id'})

pf.etable([reg_long_hte1, reg_long_hte2, reg_long_hte3, reg_long_hte4, reg_long_hte5, reg_long_hte6])

GT(_tbl_data=  level_0                      level_1                    0  \
0    coef                        treat  -0.259 <br> (0.571)   
1    coef                impulsive_num  -0.031 <br> (0.208)   
2    coef          treat:impulsive_num   0.136 <br> (0.246)   
3    coef        price_sensitivity_num                        
4    coef  treat:price_sensitivity_num                        
5      fe                      product                    x   
6   stats                 Observations                  180   
7   stats                    S.E. type      by: response_id   
8   stats                R<sup>2</sup>                0.131   
9   stats         R<sup>2</sup> Within                0.006   

                    1                    2                     3  \
0  0.051 <br> (0.521)  -0.490 <br> (0.526)   -0.273 <br> (0.534)   
1  0.124 <br> (0.150)   0.060 <br> (0.112)                         
2  0.052 <br> (0.222)   0.262 <br> (0.202)                         
3                                           -0.427* <br> (0.188)   
4                                             0.208 <br> (0.231)   
5                   x                    x                     x   
6                 180                  180                   180   
7     by: response_id      by: response_id       by: response_id   
8               0.147                0.064                 0.164   
9               0.034                0.058                 0.043   

                      4                    5  
0   -0.353 <br> (0.528)   0.553 <br> (0.653)  
1                                             
2                                             
3  -0.319* <br> (0.120)   0.032 <br> (0.229)  
4    0.296 <br> (0.207)  -0.142 <br> (0.295)  
5                     x                    x  
6                   180                  180  
7       by: response_id      by: response_id  
8                 0.148                0.020  
9                 0.034                0.014  , _body=<great_tables._gt_data.Body object at 0x11a700e10>, _boxhead=Boxhead([ColInfo(var='level_0', type=<ColInfoTypeEnum.row_group: 3>, column_label='level_0', column_align='center', column_width=None), ColInfo(var='level_1', type=<ColInfoTypeEnum.stub: 2>, column_label='level_1', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label='(1)', column_align='center', column_width=None), ColInfo(var='1', type=<ColInfoTypeEnum.default: 1>, column_label='(2)', column_align='center', column_width=None), ColInfo(var='2', type=<ColInfoTypeEnum.default: 1>, column_label='(3)', column_align='center', column_width=None), ColInfo(var='3', type=<ColInfoTypeEnum.default: 1>, column_label='(4)', column_align='center', column_width=None), ColInfo(var='4', type=<ColInfoTypeEnum.default: 1>, column_label='(5)', column_align='center', column_width=None), ColInfo(var='5', type=<ColInfoTypeEnum.default: 1>, column_label='(6)', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x11aff8210>, _spanners=Spanners([SpannerInfo(spanner_id='likelihood', spanner_level=1, spanner_label='likelihood', spanner_units=None, spanner_pattern=None, vars=['0', '3'], built=None), SpannerInfo(spanner_id='urgency', spanner_level=1, spanner_label='urgency', spanner_units=None, spanner_pattern=None, vars=['1', '4'], built=None), SpannerInfo(spanner_id='regret', spanner_level=1, spanner_label='regret', spanner_units=None, spanner_pattern=None, vars=['2', '5'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x119a2de10>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x119a2c910>, _source_notes=['Significance levels: * p < 0.05, ** p < 0.01, *** p < 0.001. Format of coefficient cell:\nCoefficient \n (Std. Error)'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x119b08160>, _formats=[], _substitution